In [2]:
import pandas as pd

# Укажите точное название вашего скачанного файла
file_path = 'Данные для тестового задания.xlsx'

# Вариант А: Загрузить конкретную вкладку по названию
df_audience = pd.read_excel(file_path, sheet_name='Данные об аудитории')
df_ab_tests = pd.read_excel(file_path, sheet_name='Данные АБ тестов')
df_listers = pd.read_excel(file_path, sheet_name='Листеры')

# Вариант Б: Загрузить все вкладки сразу в словарь
all_sheets = pd.read_excel(file_path, sheet_name=None)
# Теперь к ним можно обращаться так: all_sheets['Листеры']

In [3]:
mau = df_audience['user_id'].nunique()
print(f"MAU продукта: {mau}")


MAU продукта: 7639


In [4]:
# 1. Считаем количество уникальных пользователей для каждого дня
daily_users = df_audience.groupby('date')['user_id'].nunique()

# 2. Находим среднее значение (DAU)
dau = daily_users.mean()

print(f"DAU продукта: {round(dau)}")


DAU продукта: 560


In [5]:
# 1. Список пользователей, пришедших 1 ноября
cohort_1nov = df_audience[df_audience['date'] == '2023-11-01']['user_id'].unique()

# 2. Список пользователей, которые были активны 2 ноября
active_2nov = df_audience[df_audience['date'] == '2023-11-02']['user_id'].unique()

# 3. Находим тех, кто есть в обоих списках (вернувшиеся)
returned_users = set(cohort_1nov).intersection(set(active_2nov))

# 4. Считаем процент
retention_1d = (len(returned_users) / len(cohort_1nov)) * 100

print(f"Retention 1-го дня: {retention_1d:.1f}%")


Retention 1-го дня: 26.6%


In [6]:
# Пользователи с просмотрами
users_with_views = df_audience[df_audience['view_adverts'] > 0]['user_id'].nunique()
# Всего уникальных пользователей
total_users = df_audience['user_id'].nunique()

conversion = (users_with_views / total_users) * 100
print(f"Конверсия: {conversion:.1f}%")


Конверсия: 46.3%


In [7]:
# Сумма всех просмотров / общее кол-во уникальных пользователей
avg_views = df_audience.groupby('user_id')['view_adverts'].sum().mean()
print(f"Среднее на пользователя: {avg_views:.1f}")


Среднее на пользователя: 2.9


In [8]:
# Исходные данные из условия
total_users = 2000
detractors = 500  # критики
promoters = 1200  # сторонники
passives = 300    # нейтралы (в расчете напрямую не участвуют)

# 1. Считаем доли в процентах
promoters_share = (promoters / total_users) * 100
detractors_share = (detractors / total_users) * 100

# 2. Считаем NPS (разница долей)
nps = promoters_share - detractors_share

print(f"Доля сторонников: {promoters_share}%")
print(f"Доля критиков: {detractors_share}%")
print(f"NPS равен: {nps}%")


Доля сторонников: 60.0%
Доля критиков: 25.0%
NPS равен: 35.0%


In [9]:
from scipy import stats

# Проходим циклом по всем трем экспериментам
for i in [1, 2, 3]:
    exp_data = df_ab_tests[df_ab_tests['experiment_num'] == i]
    
    control = exp_data[exp_data['experiment_group'] == 'control']['revenue']
    test = exp_data[exp_data['experiment_group'] == 'test']['revenue']
    
    # Считаем ARPU
    arpu_control = control.mean()
    arpu_test = test.mean()
    
    # Считаем p-value (t-test)
    t_stat, p_val = stats.ttest_ind(control, test)
    
    print(f"Эксперимент №{i}:")
    print(f"ARPU Control: {arpu_control:.2f}, ARPU Test: {arpu_test:.2f}")
    print(f"p-value: {p_val:.4f}")
    print("-" * 30)


Эксперимент №1:
ARPU Control: 722.46, ARPU Test: 665.74
p-value: 0.6888
------------------------------
Эксперимент №2:
ARPU Control: 704.65, ARPU Test: 332.93
p-value: 0.0010
------------------------------
Эксперимент №3:
ARPU Control: 663.21, ARPU Test: 998.67
p-value: 0.0617
------------------------------


In [11]:
# Группируем по ID, суммируем выручку каждого юзера и берем среднее
avg_revenue = df_listers.groupby('user_id')['revenue'].sum().mean()

print(f"Средний доход на пользователя: {avg_revenue:.1f}")



Средний доход на пользователя: 156.5


In [12]:
# Оставляем только уникальных пользователей и считаем медиану их возраста
median_age = df_listers.drop_duplicates('user_id')['age'].median()

print(f"Медиана возраста: {median_age}")


Медиана возраста: 28.0


In [13]:
from statsmodels.stats.proportion import proportions_ztest

# Данные из условия
count = [1003, 1099]          # Количество платежей (успехи)
nobs = [100047501, 100001055] # Количество посетителей (выборка)

# Считаем конверсии
conv_a = count[0] / nobs[0]
conv_b = count[1] / nobs[1]

# Проводим Z-тест
z_stat, p_val = proportions_ztest(count, nobs)

print(f"Конверсия A: {conv_a:.7%}")
print(f"Конверсия B: {conv_b:.7%}")
print(f"Относительный прирост: {(conv_b/conv_a - 1)*100:.2f}%")
print(f"p-value: {p_val:.4f}")

if p_val < 0.05:
    print("Результат: Различия СТАТИСТИЧЕСКИ ЗНАЧИМЫ. Вариант B лучше.")
else:
    print("Результат: Различия НЕЗНАЧИМЫ.")


Конверсия A: 0.0010025%
Конверсия B: 0.0010990%
Относительный прирост: 9.62%
p-value: 0.0353
Результат: Различия СТАТИСТИЧЕСКИ ЗНАЧИМЫ. Вариант B лучше.
